In [3]:
DATA_YAML = "config/data.yaml" # dataset setting
MODEL_PATH = "weights/best.pt" 
PARAMS_JSON = "config/params.json" # data augmentation setting

OUTPUT_DIR = "weights/yolo_kv26"
OUTPUT_NAME = "yolo_kv26"
COMMON_XMODEL_PATH = "quantize_result/YOLO_int.xmodel"
ARCH_FILE = "../dpu_arch/DPUCZDX8G/KV260/arch.json"

In [4]:
import yaml
import os

def load_yolo_data(yaml_path):
    # Check if file exists
    if not os.path.exists(yaml_path):
        print(f"File not found: {yaml_path}")
        return None

    with open(yaml_path, 'r', encoding='utf-8') as f:
        try:
            # Use safe_load to avoid executing malicious code, which is standard practice for handling YAML
            data = yaml.safe_load(f)
            return data
        except yaml.YAMLError as exc:
            print(f"Error reading YAML: {exc}")
            return None

config = load_yolo_data(DATA_YAML)

if config:
    print("--- Dataset configuration loaded successfully ---")
    
    # 1. Get number of classes
    NUM_CLASSES = config.get('nc', 0)
    print(f"Number of classes (nc): {NUM_CLASSES}")
    
    # Get DATASET path
    DATASET_DIR = config.get('path', '')

    # 2. Get class name list
    CLASS_NAMES = config.get('names', [])
    print(f"Class names: {CLASS_NAMES}")
    
    # 3. Get train/val paths
    TRAIN_PATH = config.get('train', 'not set')
    VAL_PATH = config.get('val', 'not set')

    TRAIN_IMG_DIR = os.path.join(DATASET_DIR, TRAIN_PATH)
    VAL_IMG_DIR   = os.path.join(DATASET_DIR, VAL_PATH)

    print(f"Dataset path: {DATASET_DIR}")
    print(f"Training set path: {TRAIN_IMG_DIR}")
    print(f"Validation set path: {VAL_IMG_DIR}")

    # If names is in dict format {0: 'person', 1: 'dog'}, handle it like this
    if isinstance(CLASS_NAMES, dict):
        class_list = list(CLASS_NAMES.values())
        print(f"Converted class list: {class_list}")

--- Dataset configuration loaded successfully ---
Number of classes (nc): 4
Class names: {0: 'swimmer', 1: 'swimmer_with_life_jacket', 2: 'boat', 3: 'life_jacket'}
Dataset path: /media/jianhua/HDD 1T/dataset/SeaDronesSee_MOT
Training set path: /media/jianhua/HDD 1T/dataset/SeaDronesSee_MOT/images/train
Validation set path: /media/jianhua/HDD 1T/dataset/SeaDronesSee_MOT/images/val
Converted class list: ['swimmer', 'swimmer_with_life_jacket', 'boat', 'life_jacket']


In [5]:
import subprocess
import sys

def run_cmd(cmd:list):
    '''
    Ex: 
        cmd = [
        sys.executable, "quant_script/quant_custom.py",
        "--model_path",   "weights/best.pt",
        "--data_dir", "/home/jianhua/Desktop/dataset/SeaDronesSee_MOT",
        "--quant_mode", "calib",
        "--input_size", "640",
        "--num_classes", "4",
        "--extra_path", ".",
        ]
    '''
    
    with subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    ) as proc:
        last_line = ""
        for line in proc.stdout:
            line = line.rstrip("\n")
            # 內容一樣就跳過
            if line == last_line:
                continue
            # 進度條行：用 \r 覆寫同一行 (tqdm 套件特殊格式)
            if "it/s" in line or "s/it" in line:
                sys.stdout.write("\r" + line)
                sys.stdout.flush()
            # 一般訊息：換行輸出
            else:
                sys.stdout.write("\n" + line + "\n")
                sys.stdout.flush()
            last_line = line

## Calib (校準)

In [4]:
"""
# Step 1 — 校準
python quant_script/quant_custom.py   
    --model_path weights/best.pt   \
    --data_dir /home/jianhua/Desktop/dataset/SeaDronesSee_MOT    \
    --num_classes 4, \
    --quant_mode calib   \
    --input_size 640 \
    --extra_path .
"""

cmd = [
    sys.executable, "quant_script/quant_custom.py",
    "--model_path",   MODEL_PATH,
    "--data_dir", DATASET_DIR,
    "--num_classes", str(NUM_CLASSES),
    "--quant_mode", "calib",
    "--input_size", "640",
    "--extra_path", ".",
]

run_cmd(cmd=cmd)


[INFO] Added to sys.path: /home/jianhua/Desktop/vitis-ai-pytorch/train


  Model:      weights/best.pt

  Mode:       calib

  Input size: (3, 640, 640)

  Classes:    4

  Device:     cuda




[VAIQ_NOTE]: Loading NNDCT kernels...

[INFO] Loading checkpoint: weights/best.pt

[INFO] Loaded training checkpoint (epoch 15).

[INFO] Creating quantizer, mode: calib



[VAIQ_NOTE]: OS and CPU information:

               system --- Linux

                 node --- jianhua-MS-7D95

              release --- 6.8.0-136-generic

              version --- #136~22.04.1-Ubuntu SMP PREEMPT_DYNAMIC Fri Jul  3 16:29:11 UTC 

              machine --- x86_64

            processor --- x86_64



[VAIQ_NOTE]: Tools version information:

                  GCC --- GCC 7.5.0

               python --- 3.8.6

              pytorch --- 1.13.1+cu117

        vai_q_pytorch --- 3.5.0+60df3f1+torch1.13.1+cu117



[VAIQ_NOTE]: GPU information:

          device name --- NVIDIA GeForce RTX 4070

     device availa

## Eval mAP (test)

In [5]:
'''
# Step 2 — 評估 mAP（test）
python quant_script/quant_custom.py \
  --model_path weights/best.pt \
  --data_dir   /path/to/dataset \
  --params, config/params.json \
  --num_classes 4 \
  --quant_mode test \
  --input_size 640 \
  --extra_path . 
'''

cmd = [
    sys.executable, "quant_script/quant_custom.py",
    "--model_path", MODEL_PATH,
    "--data_dir", DATASET_DIR,
    "--params", PARAMS_JSON,
    "--num_classes", str(NUM_CLASSES),
    "--quant_mode", "test",
    "--input_size", "640",
    "--extra_path", ".",
]

run_cmd(cmd=cmd)


[INFO] Loaded params from: config/params.json

[INFO] Added to sys.path: /home/jianhua/Desktop/vitis-ai-pytorch/train


  Model:      weights/best.pt

  Mode:       test

  Input size: (3, 640, 640)

  Classes:    4

  Device:     cuda




[VAIQ_NOTE]: Loading NNDCT kernels...

[INFO] Loading checkpoint: weights/best.pt

[INFO] Loaded training checkpoint (epoch 15).

[INFO] Creating quantizer, mode: test



[VAIQ_NOTE]: OS and CPU information:

               system --- Linux

                 node --- jianhua-MS-7D95

              release --- 6.8.0-136-generic

              version --- #136~22.04.1-Ubuntu SMP PREEMPT_DYNAMIC Fri Jul  3 16:29:11 UTC 

              machine --- x86_64

            processor --- x86_64



[VAIQ_NOTE]: Tools version information:

                  GCC --- GCC 7.5.0

               python --- 3.8.6

              pytorch --- 1.13.1+cu117

        vai_q_pytorch --- 3.5.0+60df3f1+torch1.13.1+cu117



[VAIQ_NOTE]: GPU information:

          device name --

## Export xmodel

In [6]:
'''
# Step 3 — 匯出部署檔
python quant_script/quant_custom.py \
  --model_path weights/best.pt \
  --data_dir  /home/jianhua/Desktop/dataset/SeaDronesSee_MOT  \
  --params, config/params.json \
  --num_classes 4 \
  --quant_mode test \
  --input_size 640 \
  --extra_path . \
  --target DPUCZDX8G_ISA1_B4096 \
  --batch_size 1 \
  --subset_len 1 \
  --deploy
'''

cmd = [
    sys.executable, "quant_script/quant_custom.py",
    "--model_path",   MODEL_PATH,
    "--data_dir", DATASET_DIR,
    "--params", PARAMS_JSON,
    "--num_classes", str(NUM_CLASSES),
    "--quant_mode", "test",
    "--input_size", "640",
    "--extra_path", ".",
    "--target", "DPUCZDX8G_ISA1_B4096",
    "--batch_size", "1",
    "--subset_len", "1",
    "--deploy",
]

run_cmd(cmd=cmd)


[INFO] Loaded params from: config/params.json

[INFO] Added to sys.path: /home/jianhua/Desktop/vitis-ai-pytorch/train


  Model:      weights/best.pt

  Mode:       test

  Input size: (3, 640, 640)

  Classes:    4

  Device:     cuda




[VAIQ_NOTE]: Loading NNDCT kernels...

[INFO] Loading checkpoint: weights/best.pt

[INFO] Loaded training checkpoint (epoch 15).

[INFO] Creating quantizer, mode: test



[VAIQ_NOTE]: OS and CPU information:

               system --- Linux

                 node --- jianhua-MS-7D95

              release --- 6.8.0-136-generic

              version --- #136~22.04.1-Ubuntu SMP PREEMPT_DYNAMIC Fri Jul  3 16:29:11 UTC 

              machine --- x86_64

            processor --- x86_64



[VAIQ_NOTE]: Tools version information:

                  GCC --- GCC 7.5.0

               python --- 3.8.6

              pytorch --- 1.13.1+cu117

        vai_q_pytorch --- 3.5.0+60df3f1+torch1.13.1+cu117



[VAIQ_NOTE]: GPU information:

          device name --

## Export DPU's xmodel

In [6]:
import sys
import os

# 從目前 Python 路徑逆推 conda 環境根目錄
# /home/jianhua/miniconda3/envs/vai_pytorch/bin/python -> /home/jianhua/miniconda3/envs/vai_pytorch
CONDA_ENV = os.path.dirname(os.path.dirname(sys.executable))
print(f"Conda env path : {CONDA_ENV}")

Conda env path : /home/jianhua/miniconda3/envs/vai_pytorch


In [7]:
# Pytorch xmodel 轉 DPU xmodel 工具
VAI_C_XIR = os.path.join(CONDA_ENV, "bin", "vai_c_xir")

print(f"vai_c_xir tool path: {VAI_C_XIR}")
print(f"xmodel path: {COMMON_XMODEL_PATH}")

cmd = [VAI_C_XIR, 
       "-x", COMMON_XMODEL_PATH,
       "-a", ARCH_FILE,
       "-o", OUTPUT_DIR,
       "-n", OUTPUT_NAME,
       ]

run_cmd(cmd)

vai_c_xir tool path: /home/jianhua/miniconda3/envs/vai_pytorch/bin/vai_c_xir
xmodel path: quantize_result/YOLO_int.xmodel

**************************************************

* VITIS_AI Compilation - Xilinx Inc.

**************************************************

[UNILOG][INFO] Compile mode: dpu

[UNILOG][INFO] Debug mode: null

[UNILOG][INFO] Target architecture: DPUCZDX8G_ISA1_B4096_0101000046010407

[UNILOG][INFO] Graph name: YOLO, with op num: 641

[UNILOG][INFO] Begin to compile...

[UNILOG][INFO] Total device subgraph number 5, DPU subgraph number 1

[UNILOG][INFO] Compile done.

[UNILOG][INFO] The meta json is saved to "/home/jianhua/Desktop/vitis-ai-pytorch/train/weights/yolo_kv26/meta.json"

[UNILOG][INFO] The compiled xmodel is saved to "/home/jianhua/Desktop/vitis-ai-pytorch/train/weights/yolo_kv26/yolo_kv26.xmodel"

[UNILOG][INFO] The compiled xmodel's md5sum is 4f3f53bc74e80ce567ce413462dbcb10, and has been saved to "/home/jianhua/Desktop/vitis-ai-pytorch/train/weights/yo

In [8]:
import xir

# fingerprint 要跟 vivado bd 內 dpu 的 arch.json fingerprint 相同
graph = xir.Graph.deserialize(os.path.join(OUTPUT_DIR, f"{OUTPUT_NAME}.xmodel"))
root = graph.get_root_subgraph()

for sub in root.toposort_child_subgraph():
    if sub.has_attr("device") and sub.get_attr("device") == "DPU":
        print(f"Subgraph: {sub.get_name()}")
        if sub.has_attr("dpu_fingerprint"):
            fp = sub.get_attr("dpu_fingerprint")
            print(f"  Fingerprint: 0x{fp:x}")
        if sub.has_attr("DPU"):
            print(f"  Attributes: {sub.get_attrs()}")

Subgraph: subgraph_YOLO__YOLO_DarkFPN_fpn__CSP_h1__Conv_conv1__Conv2d_conv__ret_269
  Fingerprint: 0x101000046010407


## 看 xmodel 結構 & 視覺化

In [9]:
import xir
graph = xir.Graph.deserialize(os.path.join(OUTPUT_DIR,f"{OUTPUT_NAME}.xmodel"))
for sg in graph.get_root_subgraph().toposort_child_subgraph():
    for t in sg.get_output_tensors():
        print(t.name, t.dims)

YOLO__input_0_fix [1, 640, 640, 3]
YOLO__YOLO_Head_head__ret_fix [1, 20, 20, 68]
YOLO__YOLO_Head_head__ret_525_fix [1, 40, 40, 68]
YOLO__YOLO_Head_head__ret_483_fix [1, 80, 80, 68]
YOLO__YOLO_Head_head__ret_fix_ [1, 20, 20, 68]
YOLO__YOLO_Head_head__ret_525_fix_ [1, 40, 40, 68]
YOLO__YOLO_Head_head__ret_483_fix_ [1, 80, 80, 68]


In [10]:
import graphviz

XMODEL_VISUAL = os.path.join(OUTPUT_DIR, f"{OUTPUT_NAME}.dot")
XMODEL_PATH = os.path.join(OUTPUT_DIR, f"{OUTPUT_NAME}.xmodel")
cmd = [sys.executable, "quant_script/view_xmodel.py", 
       XMODEL_PATH,
       "--dot", XMODEL_VISUAL,
       "--rankdir", "LR" # 顯示架構排版
       ]
run_cmd(cmd)


# 讀取 dot 檔並渲染
with open(XMODEL_VISUAL, "r") as f:
    dot_source = f.read()

src = graphviz.Source(dot_source)
src.render(
    filename=OUTPUT_NAME,       # 檔案名稱（不含副檔名）
    directory=OUTPUT_DIR,  # 輸出目錄
    format="svg",
    cleanup=True                # 刪除中間暫存的 .gv 檔
)

# print(f"Saved to: {output_dir / (output_name + '.svg')}")

# src.view()  # 用預設程式開啟（通常是瀏覽器）



  Vitis-AI xmodel Inspector  (PyXIR / xir)


  Loading: weights/yolo_kv26/yolo_kv26.xmodel

  Loaded successfully.




  Graph Summary


  Graph name  : YOLO

  Total ops   : 324



  Op type distribution (11 unique types):

    const-fix                                x 186

    conv2d-fix                               x 87

    concat-fix                               x 20

    eltwise-fix                              x 12

    depthwise-conv2d-fix                     x 6

    pool-fix                                 x 3

    download                                 x 3

    fix2float                                x 3

    upsample-fix                             x 2

    data-fix                                 x 1

    upload                                   x 1




  Graph Input / Output Tensors




  Input tensors (0):



  Output tensors (3):

    - YOLO__YOLO_Head_head__ret_483_fix_

      shape=1x80x80x68  dtype=float32

    - YOLO__YOLO_Head_head__ret_525_fix_

      shap

'weights/yolo_kv26/yolo_kv26.svg'

In [11]:
import xir

def inspect_xmodel_fixpoints(xmodel_path):
    print(f"[*] 正在解析模型: {xmodel_path}\n")
    
    # 1. 載入並反序列化 xmodel
    try:
        graph = xir.Graph.deserialize(xmodel_path)
    except Exception as e:
        print(f"[Error] 無法載入模型: {e}")
        return

    # 2. 取得圖中所有的操作節點 (Operations)
    ops = graph.get_ops()

    # 3. 建立表格標題
    header = f"{'Op Name':<60} | {'Op Type':<15} | {'Tensor Shape':<20} | {'Fix Point'}"
    print("-" * 115)
    print(header)
    print("-" * 115)

    # 4. 遍歷每個節點並提取 Tensor 資訊
    for op in ops:
        op_name = op.get_name()
        op_type = op.get_type()
        
        try:
            # 嘗試獲取該節點的輸出 Tensor
            out_tensor = op.get_output_tensor()
            
            # 取得 Tensor 的維度大小 (例如: [1, 224, 224, 3])
            shape = str(out_tensor.dims)
            
            # 檢查 Tensor 是否包含 fix_point 屬性
            if out_tensor.has_attr("fix_point"):
                fix_point = out_tensor.get_attr("fix_point")
            else:
                fix_point = "N/A"
                
            print(f"{op_name:<60} | {op_type:<15} | {shape:<20} | {fix_point}")
            
        except Exception:
            # 某些特殊的控制節點（如 input 宣告）可能沒有標準的 output_tensor
            print(f"{op_name:<60} | {op_type:<15} | {'N/A':<20} | N/A")

    print("-" * 115)
    print(f"[*] 解析完成。共掃描了 {len(ops)} 個節點。")

if __name__ == "__main__":
    # 🌟 直接在這裡寫死你的 .xmodel 檔案路徑 🌟
    xmodel_file = "weights/yolo_kv26/yolo_kv26.xmodel" 
    
    inspect_xmodel_fixpoints(xmodel_file)

[*] 正在解析模型: weights/yolo_kv26/yolo_kv26.xmodel

-------------------------------------------------------------------------------------------------------------------
Op Name                                                      | Op Type         | Tensor Shape         | Fix Point
-------------------------------------------------------------------------------------------------------------------
YOLO__YOLO_DarkNet_net__Sequential_p5__PSA_3__Sequential_res_m__PSABlock_0__ret_255 | eltwise-fix     | [1, 20, 20, 128]     | 4
YOLO__YOLO_DarkNet_net__Sequential_p5__PSA_3__Sequential_res_m__PSABlock_0__Sequential_conv2__Conv_1__Conv2d_conv__ret_251 | conv2d-fix      | [1, 20, 20, 128]     | 5
YOLO__YOLO_DarkNet_net__Sequential_p5__PSA_3__ret_257        | concat-fix      | [1, 20, 20, 256]     | 4
YOLO__net_p5_3_conv2_conv_weight                             | const-fix       | [256, 1, 1, 256]     | 9
YOLO__net_p5_3_conv2_conv_bias                               | const-fix       | [256]           